# Naija-Switch -- LoRA fine-tune (Kaggle)

Kaggle version of the Colab notebook -- same repo, same training script,
same base model. Use this one when Colab's free GPU quota runs out:
Kaggle's free quota is **weekly** (about 30 GPU-hours/week, resets every
week) rather than Colab's tighter rolling daily limit, and needs no card.

**One-time account setup (skip if already done):**
1. Create a free account at [kaggle.com](https://www.kaggle.com).
2. `Settings -> Phone Verification` -- verify a phone number. This is
   required once before Kaggle will give you GPU or internet access on
   any notebook. No payment/card involved.

**Per-notebook setup (do this every time you open/copy this notebook):**
1. Open the settings panel on the right (or `File -> Notebook options`).
2. **Accelerator -> GPU T4 x2** (or P100, whichever is offered).
3. **Internet -> On** -- required for `git clone`, `pip install`, and
   downloading the base model from Hugging Face. Without this, the clone
   and install cells below will fail with a network error.
4. Then: `Run -> Run All` (or the "Run All" button).

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv || echo 'No GPU detected -- open the Settings panel on the right, set Accelerator to GPU T4 x2 or P100, then Run All again.'

## Why this base model

Short version: **no openly available conversational LLM is actually
pretrained on real Nigerian Pidgin.** The only public models that have
genuinely seen Pidgin text -- AfriBERTa, AfriTeVa, the MasakhaNER-tuned
RoBERTa (`arnolfokam/roberta-base-pcm`) -- are small encoder/NER/classification
models, not instruction-following chat models, so they're a poor starting
point for a chatbot. NLLB was once planned to cover Pidgin (`pcm_Latn`) but
it never shipped in the released FLORES-200 language set either.

So the practical choice -- and the one already configured in this repo's
`configs/training_config.yaml` -- is a strong general-purpose
instruction-tuned model with a good multilingual tokenizer:
**Qwen/Qwen2.5-1.5B-Instruct**.

- **Ungated.** Downloads with no token or license click-through. Llama 3
  and Gemma 2 both require accepting a license on Hugging Face first.
- **Small.** 1.5B parameters fits and fine-tunes comfortably on Kaggle's
  free GPU in a few minutes with plain LoRA -- no 4-bit/QLoRA needed at
  this size (the repo's `train_lora.py` still supports QLoRA if you swap
  to a bigger base model later).
- **Low token fragmentation on Pidgin, incidentally.** Pidgin is
  English-lexified, so most Pidgin words tokenize the same as their English
  spellings in Qwen's vocabulary -- this is the real reason fine-tuning
  isn't painful here, more than any claimed native Pidgin knowledge.
- **Already knows how to converse.** Being instruction-tuned, it can hold
  a coherent multi-turn exchange out of the box; LoRA's job here is to
  shift its *style* toward English-Pidgin mixing, not teach conversation
  from zero.

In [ ]:
import os

REPO_URL = "https://github.com/ayoolaeni/Naija-Code-Switch.git"
REPO_DIR = "/kaggle/working/Naija-Code-Switch"

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
%cd {REPO_DIR}
!git pull

In [ ]:
# Kaggle's base image already ships torch/transformers, but pin to what
# this repo expects (peft, accelerate, etc. may still be missing/older).
!pip install -q -r requirements.txt

In [ ]:
# data/processed/ and data/splits/ are gitignored on purpose (derived
# data doesn't belong in git) -- a fresh clone only has the raw sources
# under data/authored/ and data/raw/. Regenerate the processed + split
# files from those before anything below can read data/splits/*.jsonl.
!python -m src.data_pipeline.pipeline
!python -m src.data_pipeline.split

In [ ]:
for split in ["train", "val", "test"]:
    path = f"data/splits/{split}.jsonl"
    n = sum(1 for _ in open(path, encoding="utf-8"))
    print(f"{split}: {n} dialogues")

## 1. Smoke test first

Proves the training code path (LoRA injection, tokenization, loss masking,
the overfitting check) is correct, on CPU, in seconds, using a tiny
placeholder model -- before spending real GPU time on the actual base
model. Safe to skip on repeat runs once you've seen it pass.

In [ ]:
!python -m src.modeling.train_lora --smoke-test

## 2. Real fine-tune

Uses `configs/training_config.yaml` as-is: Qwen2.5-1.5B-Instruct, LoRA
rank 16 on the attention Q/V projections, learning rate 2e-4, up to 4
epochs with early stopping on validation loss.

**Honest caveat:** most of the training set is script-generated (from
`scripts/generate_synthetic_dialogues.py`), not organically collected from
many real human code-switchers as the research brief intends -- see the
data-check cell above for the current split sizes, and `README.md`'s
"Scope of this build" section for the full breakdown. This proves the
real GPU training path works end-to-end and produces a loadable adapter
that has actually learned the style in the data -- it is not the same as
training on a real-contributor-authored corpus of this size. Watch the
console for the `[overfitting-guardrail] WARNING` line at the end; with a
repetitive templated batch mixed in, some near-verbatim warnings are
still possible and not necessarily a bug.

In [ ]:
!python -m src.modeling.train_lora --config configs/training_config.yaml

## 3. Get the trained adapter out of Kaggle

Kaggle has no `google.colab.files.download()` -- instead, anything you
write under `/kaggle/working/` automatically shows up as a downloadable
output file. The cell below zips the adapter there; you then have two
ways to grab it, no extra sign-in needed for either:

- **Quick (same session):** look at the file browser panel on the right
  side of the editor (the folder icon) -- find
  `naija-switch-lora-adapter.zip` under `/kaggle/working/`, click the
  three-dot menu next to it, and choose **Download**.
- **Durable (survives closing the notebook):** click **Save Version** (top
  right) -> **Save & Run All (Commit)**. Once it finishes, open that
  version and go to its **Output** tab -- the zip is listed there with a
  download button, and stays available even after the session ends.

In [ ]:
import shutil

shutil.make_archive("/kaggle/working/naija-switch-lora-adapter", "zip", "checkpoints/naija-switch-lora")
print("Wrote /kaggle/working/naija-switch-lora-adapter.zip -- download it from the file browser panel or the Output tab after Save Version.")

## 4. Quick test -- talk to the fine-tuned model

Loads the base model plus the LoRA adapter you just trained and runs a
few code-switched prompts through it, using the same system prompt the
chat app uses.

In [ ]:
from src.modeling.inference import LocalGenerator, GenerationParams
from src.modeling.prompt_design import get_frozen_system_prompt

generator = LocalGenerator(
    base_model_id="Qwen/Qwen2.5-1.5B-Instruct",
    lora_adapter_dir="checkpoints/naija-switch-lora",
)
system_prompt = get_frozen_system_prompt()

test_inputs = [
    "Abeg how far, wetin dey happen?",
    "I don tire o, work too much today.",
    "Can you help me check my account balance?",
]

for text in test_inputs:
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": text},
    ]
    reply = generator.generate(messages, GenerationParams(max_new_tokens=80))
    print(f"User: {text}\nNaija-Switch: {reply}\n")

## Next steps

- This run proves the real GPU fine-tuning path end-to-end. The corpus
  now clears the brief's numeric 3,000-5,000 turn target, but ~97% of
  those turns are script-generated rather than from real human
  code-switchers (see `README.md`), so this is still not the brief's
  intended dataset.
- To genuinely improve the model, replace the synthetic batch with real
  contributor-authored dialogues over time: add to `data/authored/` as
  their own `*.jsonl` file (the loader in
  `src/data_pipeline/sources/authored.py` picks up any file there
  automatically), re-run `python -m src.data_pipeline.pipeline` and
  `python -m src.data_pipeline.split`, then re-run this notebook.
- **If Kaggle's weekly GPU quota also runs out** before you're done:
  AWS SageMaker Studio Lab (free, no card, T4 GPU, ~4 hours/session,
  resets daily) and Lightning AI Studios (free monthly GPU-hour
  allowance) are both card-free fallbacks worth trying -- ask if you want
  a notebook adapted for either.
- To compare base models empirically instead of by argument, run
  `python -m src.modeling.baseline_probe` (needs `HF_TOKEN` and accepting
  each gated model's license on Hugging Face for Llama 3 / Gemma 2).
- To evaluate the fine-tuned model against the unadapted baseline, set
  `LOCAL_LORA_ADAPTER_DIR=checkpoints/naija-switch-lora` and run
  `python -m src.evaluation.compare --systems mock local` (space-separated)
  against `data/splits/test.jsonl`.